# Appendix B: Review of Metric Spaces

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Appendix B, printed pp. 395-400, PDF pp. 413-418.

**Appendix Goal.** This notebook turns the metric-space review into a computational checklist for later topology and manifold chapters. By the end, a reader should be able to recognize how a distance function controls balls, open and closed sets, convergence, continuity, boundedness, compactness intuition, and completeness. The point is not to replace the definitions with pictures. The point is to make the quantifiers inspectable: every `for every epsilon` statement becomes a test about radii, neighborhoods, tail diameters, or preimages.

Appendix B is short, but it is load-bearing. Topological spaces are introduced later by abstracting the open-set behavior of metric spaces. Manifold charts are ultimately tested by continuity, homeomorphism, compactness, and local Euclidean behavior. A metric gives a familiar training ground where each of these ideas can be measured before it is generalized away.

The local `pdftotext` extraction exposed the Appendix B text on nearby physical pages because of a page-offset mismatch in the file; the assigned span above is retained as the canonical source span. The source was used only for orientation, terminology, and structure. All prose, code, diagrams, tables, and checks below are original teaching material.

## Computational Translation Guide

| Book concept | Computational representation | What to inspect |
| --- | --- | --- |
| Euclidean norm and metric | arrays of points plus distance functions | symmetry, zero diagonal, nonnegative distances, triangle inequality |
| Open and closed balls | contour sets or finite neighborhoods | how the same center and radius change shape under different metrics |
| Open subset | local radius test around each point | whether every selected point has some ball staying inside the set |
| Closed subset | complement-open test or sequence-limit test | whether convergent sequences from the subset keep their limits inside |
| Convergent sequence | tail distance to a proposed limit | tail eventually enters every chosen ball around the limit |
| Cauchy sequence | tail diameter independent of a proposed limit | points eventually bunch together even before a limit is named |
| Continuity | epsilon-delta test or open-preimage test | small domain balls map into small target balls; open sets pull back to open sets |
| Compactness intuition | finite epsilon-nets and missing-limit diagnostics | closed bounded sets can be finitely captured in Euclidean space; noncompact sets leak mass to infinity or to a missing boundary |
| Completeness | Cauchy sequences with or without limits in the space | whether the metric space contains the points its own Cauchy data demand |

## Library Routing

This appendix is metric and two-dimensional enough that durable static diagrams should do most of the teaching. `matplotlib` is used for metric balls, proof scaffolds, and compactness sketches because the geometric objects are planar and the saved PNGs should survive static course exports. `plotly` is used once for a standalone HTML sequence explorer where toggling traces and hovering over tail values helps more than a static figure. `networkx` is used for the continuity proof dependency graph because the open-preimage theorem is a statement about how definitions feed each other. `scipy.spatial.KDTree` is used in the compactness lab because finite epsilon-net coverage is naturally a nearest-neighbor query. `numpy`, `pandas`, and `sympy` handle the numeric tables and exact sanity checks. More specialized geometry stacks would add weight without exposing additional structure for this appendix.

## Visual Storyboard

1. **Metric balls are metric-dependent.** Compare balls for Euclidean, taxicab, chessboard, and discrete metrics. The invariant is not the shape but the metric axioms and the radius rule `d(x, center) < r`.
2. **Convergence and Cauchy behavior separate limit finding from tail tightening.** Plot sequences whose tails shrink, then distinguish limits that are present in the space from limits that lie outside it.
3. **Continuity has two equivalent inspection modes.** Show a concrete epsilon-delta calculation and a proof graph for the open-subset criterion.
4. **Compactness in metric spaces is finite control.** Use a finite epsilon-net on `[0,1]` and contrast it with a sequence in `(0,1)` that accumulates at a missing endpoint.
5. **Completeness is closure under Cauchy demands.** Use rational decimal approximations to `sqrt(2)` and the inherited metric on `(0,1)` to show why being Cauchy is not enough unless the space contains the limit.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/appendix-b-review-of-metric-spaces/appendix-b-review-of-metric-spaces.ipynb",
  "course_dir": "Introduction-to-Topological-Manifolds",
  "course_title": "Introduction to Topological Manifolds",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/appendix-b-review-of-metric-spaces/appendix-b-review-of-metric-spaces.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Introduction-to-Topological-Manifolds/appendix-b-review-of-metric-spaces/appendix-b-review-of-metric-spaces.ipynb",
  "notebook_title": "Appendix B: Review of Metric Spaces",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from plotly.subplots import make_subplots
from scipy.spatial import KDTree


def find_book_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, cwd / "Introduction-to-Topological-Manifolds"]
    for parent in cwd.parents:
        candidates.append(parent / "Introduction-to-Topological-Manifolds")
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "AGENTS.md").exists() and (candidate / "source_map.json").exists():
            return candidate
    raise RuntimeError("Could not find Introduction-to-Topological-Manifolds root")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (  # noqa: E402
    assert_artifacts,
    chapter_artifact_root,
    display_artifact,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.validation import image_stats  # noqa: E402

UNIT_KEY = "appendix-b-review-of-metric-spaces"
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
CHECKS = ARTIFACT_ROOT / "checks"
TABLES = ARTIFACT_ROOT / "tables"

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

visual_storyboard = [
    {
        "item": "metric-ball-comparison",
        "concept": "open and closed balls depend on the metric, not only on the underlying set",
        "representation": "four planar ball diagrams plus finite metric axiom checks",
        "library": "matplotlib, numpy, pandas",
        "artifact": "figures/metric-ball-comparison.png",
        "inspection_target": "compare the round, diamond, square, and singleton/all-or-nothing ball shapes",
        "validation": "distance matrices have symmetry, positive diagonal behavior, and triangle-inequality residual <= 0",
    },
    {
        "item": "convergence-cauchy-completion",
        "concept": "convergence names a limit; Cauchy only measures tail tightening",
        "representation": "interactive sequence traces and tail-diameter checks",
        "library": "plotly, numpy, pandas",
        "artifact": "html/convergence-cauchy-completion.html",
        "inspection_target": "hover over tail values and compare present versus missing limits",
        "validation": "tail diameters decrease below recorded thresholds",
    },
    {
        "item": "continuity-open-preimage",
        "concept": "epsilon-delta continuity is equivalent to pulling open sets back to open sets",
        "representation": "domain/codomain interval diagram and proof dependency graph",
        "library": "matplotlib, networkx, sympy",
        "artifact": "figures/continuity-open-preimage-proof-graph.png",
        "inspection_target": "trace which local ball is chosen from an open target ball",
        "validation": "computed delta sends sampled domain points into the epsilon target ball",
    },
    {
        "item": "compactness-epsilon-net",
        "concept": "compact metric sets admit finite control; noncompact examples leak to a missing endpoint",
        "representation": "finite epsilon-net coverage and missing-limit sequence sketch",
        "library": "matplotlib, scipy.spatial.KDTree",
        "artifact": "figures/compactness-epsilon-net-vs-missing-limit.png",
        "inspection_target": "read coverage radius on [0,1] and see the open-interval sequence approach 1",
        "validation": "KDTree nearest-center distances stay below epsilon on the sampled compact interval",
    },
]

storyboard_path = CHECKS / "visual-storyboard.json"
save_json({"source_span": "printed pp. 395-400, PDF pp. 413-418", "items": visual_storyboard}, storyboard_path)
display_artifact(storyboard_path)
print(f"BOOK_ROOT = {BOOK_ROOT}")
print(f"ARTIFACT_ROOT = {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")


## 1. Metrics and Balls

A metric space keeps only the distance part of Euclidean geometry. Vector addition, scalar multiplication, dot products, and coordinates may be absent. What remains is a function `d(x, y)` satisfying symmetry, positivity, and the triangle inequality. Those three axioms are strong enough to define balls, open sets, closed sets, convergence, Cauchy sequences, continuity, boundedness, and diameter.

The first visual compares four metrics on the same ambient plane or point set. In the Euclidean metric, the unit ball is round because distance is measured by the usual length. In the taxicab metric, the unit ball is a diamond because horizontal and vertical travel costs add. In the chessboard metric, the unit ball is a square because diagonal motion costs the same as the larger coordinate change. In the discrete metric, all distinct points have distance one, so a ball with radius less than one contains only its center, while a ball with radius greater than one contains every point of the sample. This is the quickest way to see why topology abstracts from metric balls: different metrics can give different local neighborhoods, and the open sets are built out of those neighborhoods.

The code also checks metric axioms on a finite sample. A finite check is not a proof for the whole plane, but it is a useful invariant scaffold. If a proposed distance function fails symmetry or the triangle inequality on a small sample, it cannot be a metric. When the sample passes, we still owe a mathematical proof, but the computation records exactly which algebraic properties the definition requires.


In [ ]:
def euclidean(p: np.ndarray, q: np.ndarray) -> float:
    return float(np.linalg.norm(p - q, ord=2))


def taxicab(p: np.ndarray, q: np.ndarray) -> float:
    return float(np.linalg.norm(p - q, ord=1))


def chessboard(p: np.ndarray, q: np.ndarray) -> float:
    return float(np.linalg.norm(p - q, ord=np.inf))


def discrete_metric(p: np.ndarray, q: np.ndarray) -> float:
    return 0.0 if np.array_equal(p, q) else 1.0


def distance_matrix(points: np.ndarray, metric) -> np.ndarray:
    n = len(points)
    return np.array([[metric(points[i], points[j]) for j in range(n)] for i in range(n)], dtype=float)


def metric_axiom_report(points: np.ndarray, metric) -> dict[str, float | bool]:
    D = distance_matrix(points, metric)
    n = len(points)
    triangle_residual = max(D[i, k] - D[i, j] - D[j, k] for i in range(n) for j in range(n) for k in range(n))
    off_diag = D + np.eye(n)
    return {
        "symmetric": bool(np.allclose(D, D.T)),
        "zero_diagonal": bool(np.allclose(np.diag(D), 0.0)),
        "positive_off_diagonal": bool(np.all(off_diag > 0.0)),
        "max_triangle_residual": float(triangle_residual),
        "diameter": float(D.max()),
    }


sample_points = np.array([
    [-1.0, 0.0],
    [0.0, 0.0],
    [0.5, 0.5],
    [1.0, -0.25],
    [1.25, 0.75],
])
metrics = {
    "euclidean": euclidean,
    "taxicab": taxicab,
    "chessboard": chessboard,
    "discrete": discrete_metric,
}
metric_reports = {name: metric_axiom_report(sample_points, metric) for name, metric in metrics.items()}

metric_table = pd.DataFrame([
    {
        "metric": "Euclidean L2",
        "formula": "sqrt(sum((x_i-y_i)^2))",
        "unit_ball_shape": "circle",
        "topological_signal": "usual local neighborhoods in R^n",
    },
    {
        "metric": "Taxicab L1",
        "formula": "sum(abs(x_i-y_i))",
        "unit_ball_shape": "diamond",
        "topological_signal": "same open sets as Euclidean on R^n, different ball geometry",
    },
    {
        "metric": "Chessboard L-infinity",
        "formula": "max(abs(x_i-y_i))",
        "unit_ball_shape": "square",
        "topological_signal": "same open sets as Euclidean on R^n, convenient boxes",
    },
    {
        "metric": "Discrete",
        "formula": "0 if x=y, else 1",
        "unit_ball_shape": "singleton for radius < 1",
        "topological_signal": "every subset is open",
    },
])
metric_table_path = TABLES / "metric-examples.csv"
save_csv(metric_table.to_dict(orient="records"), metric_table_path)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), constrained_layout=True)
theta = np.linspace(0, 2 * np.pi, 400)

axes[0].plot(np.cos(theta), np.sin(theta), color="#1f77b4", lw=2)
axes[0].fill(np.cos(theta), np.sin(theta), color="#1f77b4", alpha=0.12)
axes[0].set_title("Euclidean")
axes[0].text(0, -1.28, "$x^2+y^2<1$", ha="center")

diamond = np.array([[1, 0], [0, 1], [-1, 0], [0, -1], [1, 0]], dtype=float)
axes[1].plot(diamond[:, 0], diamond[:, 1], color="#d62728", lw=2)
axes[1].fill(diamond[:, 0], diamond[:, 1], color="#d62728", alpha=0.12)
axes[1].set_title("Taxicab")
axes[1].text(0, -1.28, "$|x|+|y|<1$", ha="center")

square = np.array([[1, 1], [-1, 1], [-1, -1], [1, -1], [1, 1]], dtype=float)
axes[2].plot(square[:, 0], square[:, 1], color="#2ca02c", lw=2)
axes[2].fill(square[:, 0], square[:, 1], color="#2ca02c", alpha=0.12)
axes[2].set_title("Chessboard")
axes[2].text(0, -1.28, "$\\max(|x|,|y|)<1$", ha="center")

finite = np.array([[-0.8, -0.3], [-0.25, 0.7], [0.0, 0.0], [0.55, 0.45], [0.9, -0.65]])
axes[3].scatter(finite[:, 0], finite[:, 1], s=70, color="#7f7f7f", label="sample")
axes[3].scatter([0], [0], s=110, color="#9467bd", label="center")
axes[3].set_title("Discrete")
axes[3].text(0, -1.28, "$B_r(c)=\\{c\\}$ if $r<1$", ha="center")

for ax in axes:
    ax.axhline(0, color="0.82", lw=0.8)
    ax.axvline(0, color="0.82", lw=0.8)
    ax.set_aspect("equal")
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.35, 1.35)
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])

fig.suptitle("Open balls are radius tests inside a chosen metric", y=1.05, fontsize=13)
metric_ball_path = FIGURES / "metric-ball-comparison.png"
save_matplotlib(fig, metric_ball_path)
plt.close(fig)

max_norm_checks = []
for n in [1, 2, 3, 5, 8]:
    rng = np.random.default_rng(100 + n)
    vectors = rng.normal(size=(60, n))
    max_norm = np.max(np.abs(vectors), axis=1)
    l2_norm = np.linalg.norm(vectors, axis=1)
    max_norm_checks.append({
        "dimension": n,
        "max_le_l2": bool(np.all(max_norm <= l2_norm + 1e-12)),
        "l2_le_sqrt_n_max": bool(np.all(l2_norm <= math.sqrt(n) * max_norm + 1e-12)),
        "largest_ratio_l2_over_max": float(np.max(l2_norm / np.maximum(max_norm, 1e-15))),
    })

metric_checks_path = CHECKS / "metric-axioms-and-balls.json"
save_json({
    "sample_point_count": int(len(sample_points)),
    "metric_axioms": metric_reports,
    "norm_inequality_B1_sample_checks": max_norm_checks,
    "finite_sample_note": "Finite tests expose failures but do not replace proofs on the full space.",
}, metric_checks_path)

display_artifact(metric_ball_path, width=940)
display(metric_table)
display_artifact(metric_checks_path)


## 2. Open, Closed, Bounded, and Diameter

Once balls are available, openness is a local containment test. A set is open when every point inside it has some positive breathing room that remains inside the set. Closedness is defined by the complement being open, but in metric spaces there is another reliable mental model: a closed set contains the limits of all convergent sequences that live in the set. This sequential view is especially useful later, because many spaces in topology are studied through sequences, paths, and limiting constructions.

Boundedness is a global diameter test. A subset is bounded when all pairwise distances are controlled by one finite number. In Euclidean space this often feels like fitting the set into a large ball, and in a metric space Appendix B records the equivalent formulations: bounded sets have finite pairwise diameter and can be placed inside some open or closed ball. That equivalence matters because the chosen center is not the main point. What matters is the existence of a single global scale that captures the set.

The next two sections separate two ideas that are often conflated. A convergent sequence is automatically Cauchy, because all sufficiently late terms are close to the limit and hence close to each other. A Cauchy sequence need not converge unless the metric space is complete. Compactness, in the metric spaces most familiar from Euclidean geometry, adds a stronger finite-control intuition: bounded closed pieces do not allow sequences to escape to infinity or to a missing boundary. Completeness focuses on Cauchy sequences; compactness controls all sequences through subsequences and all open covers through finite subcovers.


## 3. Convergence, Cauchy Tails, and Completeness

Convergence asks for a named destination. Given a candidate limit `x`, every epsilon-ball around `x` must eventually contain the sequence. Cauchy behavior is different: it asks only whether the tail of the sequence becomes small in diameter. The Cauchy test does not mention a destination. It is therefore possible for a sequence to be internally convincing, with all late terms close to each other, while the metric space itself has no point where the sequence lands.

The plot below uses three sequences. The first is `1/n` in the real line, whose limit `0` is present. The second is `1 - 1/n` viewed as a sequence in the open interval `(0,1)`. Its real-line limit is `1`, but `1` is not part of the metric space, so the sequence is Cauchy without converging in that space. The third is the sequence of terminating decimal truncations of `sqrt(2)`. Each term is rational, and the rational tail becomes arbitrarily tight in the inherited Euclidean metric. In the rational metric space `Q`, however, the demanded limit is irrational, so the space fails to contain the limit implied by its own Cauchy data.

This is the practical meaning of completeness: no Cauchy sequence should point outside the space. Complete spaces have enough points to realize the limits their metric predicts. Closed subsets of complete metric spaces inherit this property, because a convergent Cauchy sequence cannot leave a closed set at the final limiting step.


In [ ]:
n = np.arange(1, 61)
seq_to_zero = 1.0 / n
seq_open_interval = 1.0 - 1.0 / n
sqrt2_trunc = np.array([math.floor(math.sqrt(2) * 10**k) / 10**k for k in range(1, 13)], dtype=float)
sqrt2_n = np.arange(1, len(sqrt2_trunc) + 1)


def tail_diameters(values: np.ndarray) -> np.ndarray:
    return np.array([float(values[i:].max() - values[i:].min()) for i in range(len(values))])


zero_tail = tail_diameters(seq_to_zero)
open_tail = tail_diameters(seq_open_interval)
sqrt2_tail = tail_diameters(sqrt2_trunc)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    subplot_titles=("Sequence values", "Tail diameters"),
    vertical_spacing=0.14,
)
fig.add_trace(go.Scatter(x=n, y=seq_to_zero, mode="lines+markers", name="1/n in R"), row=1, col=1)
fig.add_trace(go.Scatter(x=n, y=seq_open_interval, mode="lines+markers", name="1-1/n in (0,1)"), row=1, col=1)
fig.add_trace(go.Scatter(x=sqrt2_n, y=sqrt2_trunc, mode="lines+markers", name="decimal rationals -> sqrt(2)"), row=1, col=1)
fig.add_trace(go.Scatter(x=[1, 60], y=[0, 0], mode="lines", name="limit 0 present", line=dict(dash="dot", color="#1f77b4")), row=1, col=1)
fig.add_trace(go.Scatter(x=[1, 60], y=[1, 1], mode="lines", name="limit 1 missing from (0,1)", line=dict(dash="dot", color="#d62728")), row=1, col=1)
fig.add_trace(go.Scatter(x=[1, 12], y=[math.sqrt(2), math.sqrt(2)], mode="lines", name="sqrt(2) missing from Q", line=dict(dash="dot", color="#2ca02c")), row=1, col=1)
fig.add_trace(go.Scatter(x=n, y=zero_tail, mode="lines", name="tail diameter: 1/n"), row=2, col=1)
fig.add_trace(go.Scatter(x=n, y=open_tail, mode="lines", name="tail diameter: 1-1/n"), row=2, col=1)
fig.add_trace(go.Scatter(x=sqrt2_n, y=sqrt2_tail, mode="lines+markers", name="tail diameter: decimal sqrt(2)"), row=2, col=1)
fig.update_yaxes(type="log", row=2, col=1, title_text="diameter")
fig.update_xaxes(title_text="index n", row=2, col=1)
fig.update_layout(
    title="Convergence names a limit; Cauchy behavior measures tail tightening",
    height=720,
    width=900,
    legend=dict(orientation="h", y=-0.14),
    margin=dict(l=60, r=30, t=80, b=120),
)
convergence_html_path = HTML / "convergence-cauchy-completion.html"
save_plotly_html(fig, convergence_html_path)

convergence_table = pd.DataFrame({
    "n": n[:12],
    "1_over_n": seq_to_zero[:12],
    "1_minus_1_over_n": seq_open_interval[:12],
    "tail_diameter_1_over_n": zero_tail[:12],
    "tail_diameter_open_interval": open_tail[:12],
})
convergence_table_path = TABLES / "sequence-tail-diameters.csv"
save_csv(convergence_table.to_dict(orient="records"), convergence_table_path)

convergence_checks = {
    "one_over_n_limit_present_in_R": bool(abs(seq_to_zero[-1] - 0.0) < 0.02),
    "open_interval_real_limit": 1.0,
    "open_interval_limit_is_in_space": False,
    "open_interval_tail_diameter_at_30": float(open_tail[29]),
    "sqrt2_decimal_terms_are_rational_decimals": True,
    "sqrt2_limit_squared_residual": float(abs(math.sqrt(2) ** 2 - 2.0)),
    "sqrt2_limit_is_rational": False,
    "sqrt2_tail_diameter_last": float(sqrt2_tail[-1]),
}
convergence_checks_path = CHECKS / "convergence-cauchy-completion-checks.json"
save_json(convergence_checks, convergence_checks_path)

display_artifact(convergence_html_path, width=920, height=760)
display(convergence_table.head(8))
display_artifact(convergence_checks_path)


## 4. Continuity: Epsilon-Delta and Open Preimages

Metric continuity has a local input-output form: points close to `x` in the domain must map to points close to `f(x)` in the target. The definition is asymmetric in a useful way. The target tolerance epsilon is chosen first; the domain tolerance delta is the response. If every target ball around `f(x)` can be protected by some domain ball around `x`, the map is continuous at `x`.

The open-subset criterion packages this local statement into a topological one. A function between metric spaces is continuous exactly when the preimage of every open set is open. One direction starts with an open target set `U`, picks a point `x` mapping into it, then uses the target ball around `f(x)` that stays inside `U`. Continuity supplies a domain ball around `x`, and that domain ball stays inside the preimage. The reverse direction tests the preimage of a target ball around `f(x)`. If that preimage is open and contains `x`, it contains some domain ball around `x`, which is exactly the epsilon-delta condition.

The code uses `f(t)=t^2` near `x0=1` on the interval `[0,2]`. The symbolic identity `|y^2-1| = |y-1||y+1|` explains why a small enough domain radius works. The proof graph is not a new proof; it is a map of dependencies, showing which definition supplies which radius.


In [ ]:
x0 = 1.0
epsilon = 0.40
delta = epsilon / 3.0
ys = np.linspace(max(0.0, x0 - delta), min(2.0, x0 + delta), 801)
image_errors = np.abs(ys**2 - x0**2)
max_image_error = float(image_errors.max())

y = sp.symbols("y", real=True)
symbolic_factorization = sp.factor(y**2 - 1)

fig, axes = plt.subplots(2, 1, figsize=(8, 4.8), sharex=False, constrained_layout=True)
axes[0].plot(np.linspace(0, 2, 400), np.linspace(0, 2, 400) ** 2, color="#1f77b4", lw=2)
axes[0].scatter([x0], [x0**2], color="#d62728", zorder=3)
axes[0].axvspan(x0 - delta, x0 + delta, color="#2ca02c", alpha=0.18, label="domain delta-ball")
axes[0].axhspan(x0**2 - epsilon, x0**2 + epsilon, color="#9467bd", alpha=0.15, label="target epsilon-ball")
axes[0].set_title("For f(t)=t^2 near x0=1, a small domain ball maps into the target ball")
axes[0].set_xlabel("domain point y")
axes[0].set_ylabel("f(y)")
axes[0].legend(loc="upper left")

axes[1].plot(ys, image_errors, color="#2ca02c", lw=2)
axes[1].axhline(epsilon, color="#9467bd", lw=1.5, ls="--", label="epsilon")
axes[1].set_title("Sampled image error |f(y)-f(1)| inside the chosen delta-ball")
axes[1].set_xlabel("y in B_delta(1)")
axes[1].set_ylabel("image error")
axes[1].legend(loc="upper left")

continuity_interval_path = FIGURES / "continuity-epsilon-delta-interval.png"
save_matplotlib(fig, continuity_interval_path)
plt.close(fig)

G = nx.DiGraph()
G.add_nodes_from([
    "target open set U",
    "point x in f^{-1}(U)",
    "target ball B_e(f(x)) in U",
    "continuity at x",
    "domain ball B_d(x)",
    "B_d(x) in f^{-1}(U)",
    "preimage f^{-1}(U) is open",
    "test target ball B_e(f(x))",
    "preimage of target ball is open",
    "epsilon-delta condition",
])
G.add_edges_from([
    ("target open set U", "target ball B_e(f(x)) in U"),
    ("point x in f^{-1}(U)", "target ball B_e(f(x)) in U"),
    ("target ball B_e(f(x)) in U", "continuity at x"),
    ("continuity at x", "domain ball B_d(x)"),
    ("domain ball B_d(x)", "B_d(x) in f^{-1}(U)"),
    ("B_d(x) in f^{-1}(U)", "preimage f^{-1}(U) is open"),
    ("test target ball B_e(f(x))", "preimage of target ball is open"),
    ("preimage of target ball is open", "domain ball B_d(x)"),
    ("domain ball B_d(x)", "epsilon-delta condition"),
])
pos = {
    "target open set U": (0, 2),
    "point x in f^{-1}(U)": (0, 1),
    "target ball B_e(f(x)) in U": (1.5, 1.5),
    "continuity at x": (3, 1.5),
    "domain ball B_d(x)": (4.5, 1.5),
    "B_d(x) in f^{-1}(U)": (6, 1.5),
    "preimage f^{-1}(U) is open": (7.8, 1.5),
    "test target ball B_e(f(x))": (1.5, -0.2),
    "preimage of target ball is open": (3.3, -0.2),
    "epsilon-delta condition": (6.2, -0.2),
}
fig, ax = plt.subplots(figsize=(12, 4.8), constrained_layout=True)
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=15, width=1.4, edge_color="0.35")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#f2f5f7", edgecolors="#2f4f4f", node_size=2400)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.set_title("Proof scaffold for the open-subset criterion for continuity")
ax.axis("off")
continuity_graph_path = FIGURES / "continuity-open-preimage-proof-graph.png"
save_matplotlib(fig, continuity_graph_path)
plt.close(fig)

continuity_checks_path = CHECKS / "continuity-open-preimage-checks.json"
save_json({
    "function": "f(t)=t^2 on [0,2]",
    "x0": x0,
    "epsilon": epsilon,
    "delta": delta,
    "max_sampled_image_error_inside_delta_ball": max_image_error,
    "delta_check_passes": bool(max_image_error < epsilon + 1e-12),
    "symbolic_factorization_y_squared_minus_1": str(symbolic_factorization),
    "proof_graph_nodes": int(G.number_of_nodes()),
    "proof_graph_edges": int(G.number_of_edges()),
}, continuity_checks_path)

display_artifact(continuity_interval_path, width=820)
display_artifact(continuity_graph_path, width=940)
display_artifact(continuity_checks_path)


## 5. Applied Lab: Compactness as Finite Control

Compactness is not developed in detail inside this short appendix, but the metric review is the right place to build intuition for it. In metric spaces, compactness can be felt through finite control. A compact set cannot require infinitely many unrelated local decisions at a fixed scale. In Euclidean space, the familiar theorem is that closed and bounded subsets of `R^n` are compact. Later topology chapters formulate compactness using open covers: every open cover has a finite subcover. In metric spaces, a compatible computational shadow is the finite epsilon-net: for a chosen epsilon, finitely many centers capture every point within epsilon.

The lab below samples the closed interval `[0,1]`, chooses finitely many centers, and verifies with a nearest-neighbor query that every sampled point lies within epsilon of a center. This does not prove compactness by itself, because a sample is not the full continuum. It does, however, demonstrate the kind of finite witness compactness promises. The same figure contrasts a sequence in `(0,1)` that approaches the missing endpoint `1`. The sequence is bounded and Cauchy in the inherited metric, but the limit is absent. This shows why boundedness alone is not compactness and why closedness matters in Euclidean compactness.

Use the sliders mentally: if epsilon is made smaller, `[0,1]` needs more centers but still finitely many. The open interval has no problem with boundedness, but it loses the endpoint demanded by the sequence. Compact metric spaces prevent that kind of leak: every sequence has a convergent subsequence whose limit remains in the space.


In [ ]:
epsilon_net_radius = 0.125
closed_samples = np.linspace(0.0, 1.0, 1001)
centers = np.linspace(0.0, 1.0, 6)
tree = KDTree(centers.reshape(-1, 1))
nearest_dist, nearest_idx = tree.query(closed_samples.reshape(-1, 1), k=1)
max_nearest_dist = float(nearest_dist.max())
coverage_passes = bool(max_nearest_dist <= epsilon_net_radius + 1e-12)

missing_sequence_n = np.arange(2, 80)
missing_sequence = 1.0 - 1.0 / missing_sequence_n
missing_tail = tail_diameters(missing_sequence)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
axes[0].plot(closed_samples, np.zeros_like(closed_samples), color="0.7", lw=6, solid_capstyle="round", label="[0,1]")
for c in centers:
    axes[0].plot([max(0, c - epsilon_net_radius), min(1, c + epsilon_net_radius)], [0, 0], color="#1f77b4", lw=12, alpha=0.22)
    axes[0].scatter([c], [0], color="#1f77b4", s=55, zorder=3)
axes[0].set_ylim(-0.25, 0.35)
axes[0].set_yticks([])
axes[0].set_title("Finite epsilon-net captures sampled [0,1]")
axes[0].set_xlabel("closed interval coordinate")
axes[0].legend(loc="upper center")

axes[1].scatter(missing_sequence, np.zeros_like(missing_sequence), c=missing_sequence_n, cmap="viridis", s=26)
axes[1].scatter([1.0], [0.0], s=100, facecolors="none", edgecolors="#d62728", lw=2, label="missing endpoint 1")
axes[1].annotate("sequence accumulates here", xy=(1.0, 0), xytext=(0.72, 0.2), arrowprops=dict(arrowstyle="->", color="#d62728"), color="#d62728")
axes[1].set_xlim(0.45, 1.03)
axes[1].set_ylim(-0.25, 0.35)
axes[1].set_yticks([])
axes[1].set_title("Bounded open interval leaks a limit")
axes[1].set_xlabel("points 1 - 1/n in (0,1)")
axes[1].legend(loc="upper left")

compactness_path = FIGURES / "compactness-epsilon-net-vs-missing-limit.png"
save_matplotlib(fig, compactness_path)
plt.close(fig)

compactness_rows = [
    {
        "n": int(missing_sequence_n[i]),
        "point_1_minus_1_over_n": float(missing_sequence[i]),
        "tail_diameter_from_here": float(missing_tail[i]),
    }
    for i in range(0, len(missing_sequence), 8)
]
compactness_table_path = TABLES / "compactness-missing-endpoint-tail.csv"
save_csv(compactness_rows, compactness_table_path)

compactness_checks_path = CHECKS / "compactness-epsilon-net-checks.json"
save_json({
    "epsilon_net_radius": epsilon_net_radius,
    "centers": [float(c) for c in centers],
    "sample_count_on_closed_interval": int(len(closed_samples)),
    "max_nearest_center_distance": max_nearest_dist,
    "sampled_closed_interval_covered": coverage_passes,
    "open_interval_sequence_limit_in_R": 1.0,
    "open_interval_sequence_limit_in_space": False,
    "missing_sequence_tail_diameter_last": float(missing_tail[-1]),
}, compactness_checks_path)

display_artifact(compactness_path, width=920)
display(pd.DataFrame(compactness_rows).head())
display_artifact(compactness_checks_path)


## 6. Proof and Invariant Scaffolds

Here is the compact proof logic that the visuals are meant to support.

**Metric axioms to balls.** Positivity tells us that a ball of sufficiently small radius can isolate a point in the discrete metric. Symmetry keeps the center-point distance reversible. The triangle inequality is the workhorse behind open balls being open: if `y` is inside `B_r(x)`, then any point close enough to `y` remains inside `B_r(x)` because `d(z,x) <= d(z,y) + d(y,x)`.

**Closed sets and sequences.** If a set is closed, its complement is open. A sequence from the closed set cannot converge to a point in the complement, because openness of the complement would give a ball around the alleged limit that eventually traps the sequence outside the closed set. This is the sequence-limit test used constantly in analysis and topology.

**Continuity.** The epsilon-delta definition and the open-preimage criterion are the same local radius argument read in opposite directions. Continuity converts target balls to domain balls. Openness supplies target balls inside open sets. Their composition gives open preimages.

**Completeness.** Every convergent sequence is Cauchy by the triangle inequality: late terms are each close to the limit, so they are close to each other. Completeness is the converse as a property of the space, not of a single formula. The rational and open-interval examples fail because the metric predicts a limiting point that the space does not include.

**Compactness intuition.** In metric settings, compactness should feel like no escape route remains. There is no escape to infinity, no escape to a missing boundary, and no need for infinitely many independent local choices at a fixed scale. This is why closed bounded subsets of Euclidean space are the model examples, while `(0,1)` and `R` are the standard warnings.


In [ ]:
finite_metric_points = np.array([[0.0], [0.4], [1.0], [1.7]])
finite_l1_report = metric_axiom_report(finite_metric_points, lambda p, q: float(abs(p[0] - q[0])))

eps_value = 0.1
seq = 1.0 / np.arange(1, 2000)
N = int(np.ceil(2 / eps_value)) + 1
tail = seq[N - 1:]
convergent_implies_cauchy_sample = bool(np.max(np.abs(tail[:, None] - tail[None, :])) < eps_value)

invariant_scaffold_path = CHECKS / "proof-invariant-scaffold.json"
save_json({
    "finite_real_line_l1_metric_report": finite_l1_report,
    "convergent_sequence_epsilon": eps_value,
    "chosen_N_for_1_over_n_tail": N,
    "tail_pairwise_distance_less_than_epsilon": convergent_implies_cauchy_sample,
    "triangle_inequality_role": "d(x_i,x_j) <= d(x_i,L) + d(L,x_j)",
    "closed_subset_completeness_scaffold": "Cauchy in a closed subset of a complete space converges in the ambient space, then closedness keeps the limit in the subset.",
}, invariant_scaffold_path)
display_artifact(invariant_scaffold_path)


## Final Sanity Checks

The final cell verifies three kinds of claims. First, every named artifact exists and is nonempty. Second, the generated PNGs are not blank according to simple image statistics. Third, the mathematical checks recorded during the notebook are internally consistent: sampled metrics satisfy the metric axioms, the chosen delta for the continuity example works on the sample grid, the finite epsilon-net covers the sampled interval, and the Cauchy/completeness examples record their missing limits honestly.


In [ ]:
artifact_paths = [
    storyboard_path,
    metric_ball_path,
    metric_table_path,
    metric_checks_path,
    convergence_html_path,
    convergence_table_path,
    convergence_checks_path,
    continuity_interval_path,
    continuity_graph_path,
    continuity_checks_path,
    compactness_path,
    compactness_table_path,
    compactness_checks_path,
    invariant_scaffold_path,
]
assert_artifacts(artifact_paths, min_bytes=64)

png_stats = [image_stats(path) for path in [metric_ball_path, continuity_interval_path, continuity_graph_path, compactness_path]]
for stats in png_stats:
    assert stats["bytes"] > 1000, stats
    assert stats["max_channel_stddev"] > 1.0, stats

metric_payload = json.loads(metric_checks_path.read_text(encoding="utf-8"))
for name, report in metric_payload["metric_axioms"].items():
    assert report["symmetric"], name
    assert report["zero_diagonal"], name
    assert report["positive_off_diagonal"], name
    assert report["max_triangle_residual"] <= 1e-12, name
for row in metric_payload["norm_inequality_B1_sample_checks"]:
    assert row["max_le_l2"] and row["l2_le_sqrt_n_max"], row

continuity_payload = json.loads(continuity_checks_path.read_text(encoding="utf-8"))
assert continuity_payload["delta_check_passes"]
assert continuity_payload["max_sampled_image_error_inside_delta_ball"] < continuity_payload["epsilon"] + 1e-12

convergence_payload = json.loads(convergence_checks_path.read_text(encoding="utf-8"))
assert convergence_payload["one_over_n_limit_present_in_R"]
assert convergence_payload["open_interval_limit_is_in_space"] is False
assert convergence_payload["sqrt2_limit_is_rational"] is False

compactness_payload = json.loads(compactness_checks_path.read_text(encoding="utf-8"))
assert compactness_payload["sampled_closed_interval_covered"]
assert compactness_payload["open_interval_sequence_limit_in_space"] is False

invariant_payload = json.loads(invariant_scaffold_path.read_text(encoding="utf-8"))
assert invariant_payload["finite_real_line_l1_metric_report"]["max_triangle_residual"] <= 1e-12
assert invariant_payload["tail_pairwise_distance_less_than_epsilon"]

final_sanity_path = CHECKS / "final-sanity.json"
save_json({
    "artifact_count_checked": len(artifact_paths),
    "png_stats": png_stats,
    "metric_axioms_checked": sorted(metric_payload["metric_axioms"].keys()),
    "continuity_delta_check_passes": continuity_payload["delta_check_passes"],
    "compactness_sample_covered": compactness_payload["sampled_closed_interval_covered"],
    "status": "passed",
}, final_sanity_path)
assert_artifacts([final_sanity_path], min_bytes=64)
display_artifact(final_sanity_path)
print("final_sanity: passed")


## Takeaways

A metric is a distance rule strong enough to support a full local language. Balls come first; open sets are unions of local ball decisions; closed sets can be recognized by complements and, in metric spaces, by sequence limits. The same distance rule measures boundedness and diameter.

Convergence and Cauchy behavior are related but not identical. Convergence points to a limit in the space. Cauchy behavior says that the tail becomes internally small. Completeness is the promise that every such Cauchy demand is answered by an actual point of the space.

Continuity can be read at two resolutions. The epsilon-delta definition is pointwise and metric. The open-preimage criterion is setwise and topological. Their equivalence is one of the main bridges from analysis to topology.

Compactness should be remembered as finite control. In Euclidean metric spaces, closed and bounded sets are the model compact sets; missing endpoints and unbounded directions are the basic ways compactness fails. Later chapters abstract this finite-control idea away from formulas for distance, but the metric examples here remain the safest intuition.
